## Ambil Api Key

In [59]:
import httpx
url = "http://10.12.1.35:8585/apikey"

payload = {
    "username": "MaxDelta",
    "otp": "561930"
}

try:
    with httpx.Client(timeout=10.0) as client:
        response = client.post(url, json=payload)
        response.raise_for_status()
        data = response.json()
        
    print("Response:", data)
    
    api_key = data.get("api_key")
    print("API Key:", api_key)

except httpx.HTTPStatusError as e:
    print("HTTP Error:", e.response.status_code, e.response.text)

except httpx.RequestError as e:
    print("Request Error:", str(e))

Response: {'api_key': 'pr-GLv7n8xrOztYJNz-xxhhSIYotbwqEyIMLPs0VKWEb4A'}
API Key: pr-GLv7n8xrOztYJNz-xxhhSIYotbwqEyIMLPs0VKWEb4A


## Ambil Dataset

In [ ]:
from datasets import load_dataset

test_dataset = load_dataset(
    "fahadh4ilyas/indonesian_news_datasets",
    split="test",
    token="hf_xxxxxxxxxxxxx" 
)

test_dataset

Dataset({
    features: ['id', 'source', 'title', 'image', 'url', 'content', 'date', 'embedding', 'created_at', 'updated_at', 'summary', 'nomic_embedding', 'label'],
    num_rows: 6459
})

In [61]:
import pandas as pd

test_df = pd.DataFrame(test_dataset)
test_df.head()

,id,source,title,image,url,content,date,embedding,created_at,updated_at,summary,nomic_embedding,label
0,3769,cnnindonesia,"Sidak di Kantor Pajak Solo, Jokowi Pamer Setor...",https://akcdn.detik.net.id/visual/2023/02/06/p...,https://www.cnnindonesia.com/nasional/20230309...,Presiden JokoWidodo( Jokowi )melakukan inspeks...,2023-03-09 10:23:49,"[-0.006528483,-0.016078303,0.019445695,-0.0227...",2023-03-09 11:05:15.323051,2023-03-09 11:05:15.323051,Presiden Jokowi bersama dengan Menteri Keuanga...,"[-0.27555179595947266, 0.7974510788917542, -2....",POLITIK_PEMERINTAHAN
1,36770,kumparan,5 Contoh Soal Advertisement Kelas 9 untuk Bela...,https://blue.kumparan.com/image/upload/fl_prog...,https://kumparan.com/berita-terkini/5-contoh-s...,Advertisement merupakan salah satu materi dala...,2023-03-27 12:15:11,"[0.006514681,0.011406955,0.005211745,-0.017439...",2023-03-27 12:20:12.787908,2023-03-27 12:20:12.787908,Materi advertisement mengenalkan siswa tentang...,"[0.14575253427028656, 1.7195326089859009, -2.4...",EKONOMI_BISNIS
2,32538,kumparan,Dimensi Sosial dalam Puasa Ramadhan,https://blue.kumparan.com/image/upload/fl_prog...,https://kumparan.com/faisal-dudayef15397841851...,Puasa Ramadhan adalah salah satu ibadah pentin...,2023-03-25 11:48:05,"[-0.009245157,-0.009270434,0.020424023,-0.0065...",2023-03-25 12:20:16.571114,2023-03-25 12:20:16.571114,Puasa Ramadhan diwajibkan bagi umat Muslim dan...,"[0.8499172329902649, 0.29111120104789734, -2.2...",POLITIK_PEMERINTAHAN
3,9979,kumparan,"Jalur Rel Terkena Longsor, Perjalanan Kereta P...",https://blue.kumparan.com/image/upload/fl_prog...,https://kumparan.com/kumparannews/jalur-rel-te...,Longsor terjadi di sekitar area jalur rek kere...,2023-03-15 00:56:19,"[0.007497486,-0.029989945,0.012072223,-7.43018...",2023-03-15 01:50:15.84186,2023-03-15 01:50:15.84186,Longsor terjadi di jalur Kereta Api Pangrango ...,"[0.6692487597465515, 0.9170069098472595, -2.81...",BENCANA_LINGKUNGAN
4,29714,tempo,"Ada Trojan Perbankan Versi Baru, Kaspersky Seb...",https://statik.tempo.co/data/2020/04/18/id_931...,https://bisnis.tempo.co/read/1706385/ada-troja...,"TEMPO.CO, Jakarta - Pakar dari Kaspersky telah...",2023-03-24 02:02:21,"[-0.020842083,-0.031541903,0.013520802,-0.0256...",2023-03-24 02:30:19.971467,2023-03-24 02:30:19.971467,"Emotet, program malware komputer yang awalnya ...","[0.22268828749656677, 1.3627448081970215, -2.8...",TEKNOLOGI_DIGITAL


## Engine Klasifikasi News

In [134]:
import httpx
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

BASE_URL = "http://10.12.1.35:8585"
MODEL_NAME = "Qwen3-4B"
MAX_WORKERS = 10
SUMMARY_MAX_CHARS = 4000
OUTPUT_FILE = "hasil_klasifikasi_news.csv"

VALID_LABELS = {
    "POLITIK_PEMERINTAHAN",
    "EKONOMI_BISNIS",
    "HUKUM_KRIMINAL",
    "OLAHRAGA",
    "TEKNOLOGI_DIGITAL",
    "BENCANA_LINGKUNGAN"
}

SYSTEM_PROMPT = (
    "You are a strict Indonesian news classification system. "
    "Your only task is to read a news article and output exactly one of these 6 labels: "
    "POLITIK_PEMERINTAHAN, EKONOMI_BISNIS, HUKUM_KRIMINAL, OLAHRAGA, TEKNOLOGI_DIGITAL, BENCANA_LINGKUNGAN. "
    "Never output anything else. No explanations, no punctuation, no extra text. Just the label."
)

CLASSIFICATION_PROMPT = """Classify the following Indonesian news article into exactly ONE of these 6 labels:

POLITIK_PEMERINTAHAN
EKONOMI_BISNIS
HUKUM_KRIMINAL
OLAHRAGA
TEKNOLOGI_DIGITAL
BENCANA_LINGKUNGAN

CATEGORY DEFINITIONS:
- POLITIK_PEMERINTAHAN: news about how a country or region is governed and managed through political processes, including government decisions, policies, elections, diplomacy, and state institutions
- EKONOMI_BISNIS: news about the production, distribution, and consumption of goods and services, including financial markets, trade, investment, and business activities
- HUKUM_KRIMINAL: news about the system of rules, enforcement of laws, criminal acts, and the judicial process
- OLAHRAGA: news where the PRIMARY topic is an ongoing or completed sports competition, athlete performance result, or sporting event outcome — not a sports figure's non-sport activity, not sports business, not sports policy
- TEKNOLOGI_DIGITAL: news where the PRIMARY topic is a technology product, digital platform, or tech innovation — not merely mentioning a tool or app in passing
- BENCANA_LINGKUNGAN: news where the PRIMARY topic is a natural disaster (flood, earthquake, landslide, eruption, wildfire) or environmental damage actively occurring — NOT accidents, crimes, or human-caused incidents

DISAMBIGUATION RULES:

POLITIK vs EKONOMI:
- Ask: is the main actor a government/state institution making a political decision? → POLITIK_PEMERINTAHAN
- Ask: is the main actor a company, market, or private sector? → EKONOMI_BISNIS
- Government issues policy or regulation → POLITIK_PEMERINTAHAN
- State budget (APBN/APBD) approval or debate → POLITIK_PEMERINTAHAN
- Company earnings, stock market, private investment, business merger → EKONOMI_BISNIS
- Trade data, inflation, GDP, economic indicators (reported, not decided) → EKONOMI_BISNIS
- Government economic program where the policy decision is the focus → POLITIK_PEMERINTAHAN
- Government economic program where the economic impact/data is the focus → EKONOMI_BISNIS
- News about prices, commodities, supply chain, exports, imports → EKONOMI_BISNIS even if government is mentioned
- News about banks, insurance, fintech, capital markets → EKONOMI_BISNIS even if regulated by government
- News about UMKM, entrepreneurs, cooperatives, industry sectors → EKONOMI_BISNIS

HUKUM_KRIMINAL takes priority over BENCANA_LINGKUNGAN:
- Victims or casualties caused by human action (crime, negligence, accident, violence) → HUKUM_KRIMINAL
- Fire caused by humans or unknown cause → HUKUM_KRIMINAL (not BENCANA)
- Building collapse due to construction fault → HUKUM_KRIMINAL (not BENCANA)
- Traffic accident with casualties → HUKUM_KRIMINAL (not BENCANA)
- Corruption, bribery, fraud, money laundering, KPK, police arrest, court verdict → HUKUM_KRIMINAL (never POLITIK)
- Government official arrested or on trial → HUKUM_KRIMINAL (not POLITIK, even if they are a politician)

BENCANA_LINGKUNGAN only when:
- The natural disaster itself (flood, earthquake, landslide, volcanic eruption, tsunami, wildfire from nature) is the PRIMARY topic
- Includes: scale of damage, number of displaced people, rescue operations in progress
- BMKG weather warning where the weather event is the main focus
- Government response or policy about a disaster → POLITIK_PEMERINTAHAN (not BENCANA)
- Disaster relief budget → EKONOMI_BISNIS (not BENCANA)

OLAHRAGA only when:
- The match, competition, tournament, or athlete performance IS the main topic
- Includes: match results, standings, player transfers, training camp, injury news
- Does NOT include: sports corruption → HUKUM_KRIMINAL, sports funding policy → POLITIK_PEMERINTAHAN

TEKNOLOGI_DIGITAL:
- Tech mentioned only as context → use the more appropriate label instead
- Tech regulation by government → TEKNOLOGI_DIGITAL (not POLITIK)

OUT-OF-SCOPE MAPPING:
- Entertainment, film, music, celebrity → EKONOMI_BISNIS
- Religion, culture, tourism as industry → EKONOMI_BISNIS
- Lifestyle, food, travel (non-policy) → EKONOMI_BISNIS
- Trivia, crossword, quiz, games → TEKNOLOGI_DIGITAL
- Science, research, innovation → TEKNOLOGI_DIGITAL
- Education policy, school regulation, national curriculum → POLITIK_PEMERINTAHAN
- Social welfare policy, poverty alleviation program → POLITIK_PEMERINTAHAN
- Public health policy, hospital regulation, national health program → POLITIK_PEMERINTAHAN

CRITICAL RULES:
- You MUST output ONLY one of the 6 labels above, nothing else.
- Do NOT output any label outside the 6 above under any circumstance.
- Do NOT write explanations, reasoning, or punctuation.
- Even if the article is completely unrelated, you MUST still choose the closest one from the 6 labels.

Title: {title}
Content: {content}

Label:"""


def normalize_label(raw: str) -> str | None:
    cleaned = raw.strip().upper().replace(" ", "_").replace("-", "_")
    if cleaned in VALID_LABELS:
        return cleaned
    for label in VALID_LABELS:
        if cleaned in label or label.split("_")[0] == cleaned:
            return label
    return None


def classify_news(title: str, content: str, api_key: str) -> tuple[str | None, str | None]:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": CLASSIFICATION_PROMPT.format(title=title, content=content)}
        ],
        "temperature": 0.1,
        "top_p": 0.9,
        "max_tokens": 20,
        "chat_template_kwargs": {"enable_thinking": False}
    }

    with httpx.Client(timeout=60.0) as client:
        response = client.post(f"{BASE_URL}/v1/chat/completions", json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()

    raw_label = data["choices"][0]["message"]["content"].strip()
    return normalize_label(raw_label), data.get("id")


def process_row(row: dict, api_key: str) -> dict:
    try:
        predicted_label, response_id = classify_news(
            title=row["title"],
            content=row["content"][:SUMMARY_MAX_CHARS],
            api_key=api_key
        )
        return {**row, "predicted_label": predicted_label, "response_id": response_id, "error": None}
    except Exception as e:
        return {**row, "predicted_label": None, "response_id": None, "error": str(e)}


def run_batch(rows: list[dict], api_key: str) -> list[dict]:
    total = len(rows)
    results = [None] * total
    counter = {"done": 0}
    lock = Lock()

    print(f"Memproses {total} artikel dengan {MAX_WORKERS} workers...\n")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_idx = {
            executor.submit(process_row, row, api_key): idx
            for idx, row in enumerate(rows)
        }
        for future in as_completed(future_to_idx):
            idx = future_to_idx[future]
            result = future.result()
            results[idx] = result

            with lock:
                counter["done"] += 1
                done = counter["done"]

            status = f"[ERROR: {result['error']}]" if result["error"] else result["predicted_label"]
            print(f"({done}/{total}) ID: {result.get('id', '?')} → {status}")

    return results

In [135]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

rows = list(test_dataset)
total = len(rows)
results = [None] * total
counter = {"done": 0}
lock = Lock()

def process_row(idx: int, row: dict):
    try:
        predicted_label, response_id = classify_news(
            title=row["title"],
            content=row["content"][:4000],
            api_key=api_key
        )
        results[idx] = {
            "id": row["id"],
            "label": row["label"],
            "predicted_label": predicted_label,
            "response_id": response_id,
            "error": None
        }
    except Exception as e:
        results[idx] = {
            "id": row["id"],
            "label": row["label"],
            "predicted_label": None,
            "response_id": None,
            "error": str(e)
        }
    finally:
        with lock:
            counter["done"] += 1
            done = counter["done"]
        status = results[idx]["predicted_label"] or f"ERROR: {results[idx]['error']}"
        print(f"({done}/{total}) ID: {row['id']} → {status}")

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(process_row, idx, row) for idx, row in enumerate(rows)]
    for future in as_completed(futures):
        future.result()

print(f"\nSelesai. Sukses: {sum(1 for r in results if r['predicted_label'])} | Error: {sum(1 for r in results if r['error'])}")

(1/6459) ID: 24660 → HUKUM_KRIMINAL
(2/6459) ID: 3769 → EKONOMI_BISNIS
(3/6459) ID: 32538 → POLITIK_PEMERINTAHAN
(4/6459) ID: 52720 → HUKUM_KRIMINAL
(5/6459) ID: 4765 → HUKUM_KRIMINAL
(6/6459) ID: 29714 → TEKNOLOGI_DIGITAL
(7/6459) ID: 9979 → BENCANA_LINGKUNGAN
(8/6459) ID: 36770 → EKONOMI_BISNIS
(9/6459) ID: 52849 → POLITIK_PEMERINTAHAN
(10/6459) ID: 16192 → BENCANA_LINGKUNGAN
(11/6459) ID: 58657 → OLAHRAGA
(12/6459) ID: 23291 → HUKUM_KRIMINAL
(13/6459) ID: 61426 → POLITIK_PEMERINTAHAN
(14/6459) ID: 37597 → EKONOMI_BISNIS
(15/6459) ID: 22833 → EKONOMI_BISNIS
(16/6459) ID: 50796 → POLITIK_PEMERINTAHAN
(17/6459) ID: 37999 → HUKUM_KRIMINAL
(18/6459) ID: 16546 → EKONOMI_BISNIS
(19/6459) ID: 61901 → EKONOMI_BISNIS
(20/6459) ID: 48278 → POLITIK_PEMERINTAHAN
(21/6459) ID: 26214 → POLITIK_PEMERINTAHAN
(22/6459) ID: 24987 → POLITIK_PEMERINTAHAN
(23/6459) ID: 56363 → HUKUM_KRIMINAL
(24/6459) ID: 19294 → POLITIK_PEMERINTAHAN
(25/6459) ID: 33039 → EKONOMI_BISNIS
(26/6459) ID: 9562 → HUKUM_KRIMINA

In [136]:
import pandas as pd

df_result = pd.DataFrame(results)
df_result.to_csv("hasil_klasifikasi_news.csv", index=False)

total = len(df_result)
sukses = df_result["predicted_label"].notna().sum()
error = df_result["error"].notna().sum()
akurasi = (df_result["label"] == df_result["predicted_label"]).mean() * 100

print(f"Total   : {total}")
print(f"Sukses  : {sukses}")
print(f"Error   : {error}")
print(f"Akurasi : {akurasi:.2f}%")
print(f"File    : hasil_klasifikasi_news.csv")

Total   : 6459
Sukses  : 6459
Error   : 0
Akurasi : 83.34%
File    : hasil_klasifikasi_news.csv


In [137]:
df = pd.read_csv("hasil_klasifikasi_news.csv")
label_percentage = df["label"].value_counts(normalize=True) * 100
print("Distribusi Label (%):")
print(label_percentage.round(2))

Distribusi Label (%):
label
POLITIK_PEMERINTAHAN    30.78
HUKUM_KRIMINAL          27.33
EKONOMI_BISNIS          26.58
BENCANA_LINGKUNGAN       8.28
OLAHRAGA                 4.77
TEKNOLOGI_DIGITAL        2.26
Name: proportion, dtype: float64


In [138]:
predicted_percentage = df["predicted_label"].value_counts(normalize=True) * 100
print("Distribusi Predicted Label (%):")
print(predicted_percentage.round(2))

Distribusi Predicted Label (%):
predicted_label
POLITIK_PEMERINTAHAN    30.08
EKONOMI_BISNIS          27.14
HUKUM_KRIMINAL          25.56
BENCANA_LINGKUNGAN       8.92
OLAHRAGA                 6.12
TEKNOLOGI_DIGITAL        2.18
Name: proportion, dtype: float64


In [139]:
summary = pd.DataFrame({
    "Label (n)": df_result["label"].value_counts(),
    "Predicted (n)": df_result["predicted_label"].value_counts(),
    "Label (%)": df_result["label"].value_counts(normalize=True) * 100,
    "Predicted (%)": df_result["predicted_label"].value_counts(normalize=True) * 100,
}).fillna(0).round(2)

summary["Gap (%)"] = (summary["Predicted (%)"] - summary["Label (%)"]).round(2)
summary["Gap (n)"] = (summary["Predicted (n)"] - summary["Label (n)"]).astype(int)
summary = summary.sort_values("Label (%)", ascending=False)

total_row = pd.DataFrame({
    "Label (%)": [summary["Label (%)"].sum()],
    "Predicted (%)": [summary["Predicted (%)"].sum()],
    "Label (n)": [summary["Label (n)"].sum()],
    "Predicted (n)": [summary["Predicted (n)"].sum()],
    "Gap (%)": [summary["Gap (%)"].abs().sum().round(2)],
    "Gap (n)": [summary["Gap (n)"].abs().sum()]
}, index=["TOTAL"])

summary = pd.concat([summary, total_row])

summary[["Label (n)", "Predicted (n)", "Gap (n)"]] = summary[["Label (n)", "Predicted (n)", "Gap (n)"]].astype(int)

summary.style.background_gradient(cmap="RdYlGn_r", subset=["Gap (%)", "Gap (n)"])

,Label (n),Predicted (n),Label (%),Predicted (%),Gap (%),Gap (n)
POLITIK_PEMERINTAHAN,1988,1943,30.780000,30.080000,-0.700000,-45
HUKUM_KRIMINAL,1765,1651,27.330000,25.560000,-1.770000,-114
EKONOMI_BISNIS,1717,1753,26.580000,27.140000,0.560000,36
BENCANA_LINGKUNGAN,535,576,8.280000,8.920000,0.640000,41
OLAHRAGA,308,395,4.770000,6.120000,1.350000,87
TEKNOLOGI_DIGITAL,146,141,2.260000,2.180000,-0.080000,-5
TOTAL,6459,6459,100.000000,100.000000,5.100000,328


In [140]:
# Filter data yang salah prediksi
df_errors = df_result[df_result["label"] != df_result["predicted_label"]].copy()

print(f"Total salah prediksi: {len(df_errors)} dari {len(df_result)} ({len(df_errors)/len(df_result)*100:.2f}%)")
print(f"Total benar prediksi: {len(df_result) - len(df_errors)}\n")

# Tampilkan pasangan error terbanyak
error_pairs = (
    df_errors.groupby(["label", "predicted_label"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Top misclassification pairs:")
display(error_pairs)

Total salah prediksi: 1076 dari 6459 (16.66%)
Total benar prediksi: 5383

Top misclassification pairs:


,label,predicted_label,count
8,EKONOMI_BISNIS,POLITIK_PEMERINTAHAN,153
21,POLITIK_PEMERINTAHAN,EKONOMI_BISNIS,143
13,HUKUM_KRIMINAL,POLITIK_PEMERINTAHAN,128
23,POLITIK_PEMERINTAHAN,OLAHRAGA,96
20,POLITIK_PEMERINTAHAN,BENCANA_LINGKUNGAN,73
22,POLITIK_PEMERINTAHAN,HUKUM_KRIMINAL,70
0,BENCANA_LINGKUNGAN,EKONOMI_BISNIS,53
10,HUKUM_KRIMINAL,BENCANA_LINGKUNGAN,49
18,OLAHRAGA,POLITIK_PEMERINTAHAN,46
9,EKONOMI_BISNIS,TEKNOLOGI_DIGITAL,42


In [ ]:
# Load dataset asli untuk ambil title dan content
from datasets import load_dataset

raw_dataset = load_dataset(
    "fahadh4ilyas/indonesian_news_datasets",
    split="test",
    token="hf_xxxxxxxxxxxx"
)

df_raw = pd.DataFrame(raw_dataset)

# Merge dengan hasil prediksi
df_merged = df_result.merge(df_raw[["id", "title", "content"]], on="id", how="left")

# Filter yang salah
df_errors = df_merged[df_merged["label"] != df_merged["predicted_label"]].copy()

print(f"Total salah prediksi: {len(df_errors)} dari {len(df_merged)}")
print(f"Kolom tersedia: {df_errors.columns.tolist()}")

Total salah prediksi: 1076 dari 6459
Kolom tersedia: ['id', 'label', 'predicted_label', 'response_id', 'error', 'title', 'content']


In [142]:
# Investigasi per pasangan
target_ids = {
    ("EKONOMI_BISNIS", "POLITIK_PEMERINTAHAN"): [14870, 10489, 61665],
    ("POLITIK_PEMERINTAHAN", "EKONOMI_BISNIS"): [3769, 37597, 30899],
    ("HUKUM_KRIMINAL", "POLITIK_PEMERINTAHAN"): [26053, 5179, 38039],
    ("POLITIK_PEMERINTAHAN", "OLAHRAGA"): [60219, 56577, 11083],
    ("POLITIK_PEMERINTAHAN", "HUKUM_KRIMINAL"): [30508, 60164],
}

pd.set_option("display.max_colwidth", 500)

for (true_label, pred_label), ids in target_ids.items():
    print(f"\n{'='*60}")
    print(f"TRUE: {true_label} → PREDICTED: {pred_label}")
    print('='*60)

    df_sample = df_errors[df_errors["id"].isin(ids)][["id", "label", "predicted_label", "title", "content"]]
    
    for _, row in df_sample.iterrows():
        print(f"\nID    : {row['id']}")
        print(f"TITLE : {row['title']}")
        print(f"CONTENT: {str(row['content'])[:500]}")
        print("-" * 40)


TRUE: EKONOMI_BISNIS → PREDICTED: POLITIK_PEMERINTAHAN

ID    : 14870
TITLE : Kemendagri Beri APBD Award 2023 pada Daerah dengan Realisasi APBD Tertinggi
CONTENT: Suara.com - Direktur Jenderal (Dirjen) Bina Keuangan Daerah (Keuda) Kemendagri Agus Fatoni mengatakan, APBD Award 2023 diberikan kepada kepala daerah dengan tiga kategori. Pertama, daerah dengan realisasi pendapatan tertinggi. Kedua, daerah dengan realisasi belanja tertinggi. Ketiga, daerah dengan realisasi peningkatan Pendapatan Asli Daerah (PAD) tertinggi. Masing-masing kategori penghargaan tersebut diberikan kepada lima provinsi, lima kabupaten, dan lima kota. "Penilaian didasarkan dari perhi
----------------------------------------

ID    : 10489
TITLE : Salurkan 21 Juta Beras untuk Bansos Lebaran, Bulog Tunggu Data dari Kemensos
CONTENT: TEMPO.CO, Jakarta -Perusahaan Umum atau Perum Bulog menunggu data dari Kementerian Sosial atau Kemensos untuk menyalurkan beras yang menjadi bantuan sosial atau bansos Lebaran.Hal ini d